# CDK2単独の構造・活性データ横断解析 (`CDK2_HUMAN`, UniProt `P24941`)

既存の`cdk9_cdk2_split.ipynb`はCDK9/CDK2の9構造限定で`chem.protein.split()`の機能確認、
`cdk_paralogs_active_compounds.ipynb`はCDK2を含む6パラログ横断でのChEMBL高活性化合物収集
だった。本ノートブックはCDK2単独に焦点を絞り、次の3つを1つの流れとしてまとめる。

1. **構造ランドスケープ** -- 単純な解像度フィルタではなく、(a) 分子量200〜700の非溶媒/非イオン
   /非天然ヌクレオチドリガンドが結合している(=結晶化添加剤やATP/ADPそのものではなく、実際の
   低分子阻害剤が結合している)こと、(b) その構造を報告した論文の被引用数(NCBI iCite経由)、
   の2軸でCDK2のPDB全構造を評価し、「過去の研究でよく参照されてきた阻害剤結合構造」を優先的に
   収集する。
2. **ポケット** -- 被引用数上位の構造で`chem.protein.find_pocket`を実行し(自動検出ではなく、
   1で特定した実際の阻害剤コードを明示的に指定)、複数構造にわたってポケット裏打ち残基がどれ
   だけ保存されているかを確認する。
3. **活性データとの突合** -- `chem.chembl`から取得したCDK2の高活性化合物と、実際に共結晶化
   されているリガンドのコードを、正準SMILES一致で突き合わせる。

## 1. 構造ランドスケープ

### 参考情報: 全登録構造の解像度分布

まず参考情報として、解像度だけで見た全体分布を確認しておく(後述の通り、本ノートブックでは
これ自体は選定基準にしない)。公開APIには「ダウンロードせずに解像度だけ調べる」関数が無いので、
`cdk20_similar_targets.ipynb`と同様に`chem.rcsb.fetch`の内部関数(`_search_entry_ids`でCDK2に
アノテーションされた全PDBエントリIDを取得、`_fetch_resolutions`で解像度をバッチ取得)をそのまま
使う。

In [ ]:
from chem.rcsb.fetch import _fetch_resolutions, _search_entry_ids

CDK2_UNIPROT = "P24941"  # CDK2_HUMAN

all_entry_ids = _search_entry_ids(CDK2_UNIPROT)
all_resolutions = _fetch_resolutions(all_entry_ids)
print(f"{len(all_entry_ids)} PDB entries annotated with {CDK2_UNIPROT}")
print(f"{sum(1 for v in all_resolutions.values() if v is not None)} of those have a reported resolution")

In [ ]:
import matplotlib.pyplot as plt

resolved = [v for v in all_resolutions.values() if v is not None]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(resolved, bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("resolution (Å)")
ax.set_ylabel("n structures")
ax.set_title(f"CDK2_HUMAN: resolution distribution across {len(resolved)} PDB entries (reference only)")
plt.tight_layout()
plt.show()

### 各構造の非高分子リガンドと、報告論文のPubMed IDを一括取得する

`chem.rcsb.download_structures`で1件ずつ構造ファイルをダウンロードしてから中身を調べるのは、
522件全部が対象だと重い。RCSB GraphQL APIの`entries`クエリは、構造ファイルを落とさずに
(1) `nonpolymer_entities`で構造中の非高分子(HETATM由来)コンポーネントのコード一覧、
(2) `rcsb_primary_citation`でその構造を報告した論文のPubMed ID/タイトル/出版年、をまとめて
返してくれるので、これを522件バッチで取得する。

In [ ]:
import requests
from tqdm import tqdm

RCSB_GRAPHQL_API = "https://data.rcsb.org/graphql"
_BATCH_SIZE = 200

_ENTRY_QUERY = """
query($ids: [String!]!) {
    entries(entry_ids: $ids) {
        rcsb_id
        rcsb_primary_citation {
            pdbx_database_id_PubMed
            title
            rcsb_journal_abbrev
            year
        }
        nonpolymer_entities {
            pdbx_entity_nonpoly { comp_id name }
        }
    }
}
"""

entry_nonpoly_codes = {}
entry_citation = {}
batches = [all_entry_ids[i : i + _BATCH_SIZE] for i in range(0, len(all_entry_ids), _BATCH_SIZE)]
for batch in tqdm(batches, desc="fetching nonpolymer/citation metadata", unit="batch"):
    resp = requests.post(RCSB_GRAPHQL_API, json={"query": _ENTRY_QUERY, "variables": {"ids": batch}}, timeout=60)
    resp.raise_for_status()
    for entry in resp.json()["data"]["entries"]:
        entry_id = entry["rcsb_id"]
        entry_nonpoly_codes[entry_id] = sorted(
            {e["pdbx_entity_nonpoly"]["comp_id"] for e in entry["nonpolymer_entities"] or []}
        )
        entry_citation[entry_id] = entry["rcsb_primary_citation"] or {}

print(f"{sum(1 for c in entry_nonpoly_codes.values() if c)} of {len(all_entry_ids)} entries have "
      f"at least one non-polymer component")

### 「実際の阻害剤」を分子量フィルタで判定する

`chem.protein.SOLVENT_AND_IONS`(結晶化添加剤・イオン)を除外するだけでは、CDK2の天然基質
であるATP/ADP(それ自身は"阻害剤"ではない)が残ってしまう。そこで追加で

- `NATURAL_NUCLEOTIDES` -- ATP/ADP/AMPおよびその非加水分解アナログ(ANP=AMP-PNP、
  ACP=AMP-PCP、AGS=ATP-γ-S)、GTP/GDP/GNP(まれに結合)を除外する。
- 分子量200〜700(RCSB GraphQLの`chem_comps`クエリで構造ファイルを落とさず取得できる
  `formula_weight`を使う)の範囲に収まるものだけを「本当の阻害剤候補」とする。

の2条件を追加する。両方を満たすコードが1つ以上残る構造を「阻害剤結合(holo)構造」とみなす。

In [ ]:
from chem.protein import SOLVENT_AND_IONS

# ATP-competitive pocketの天然リガンド/非加水分解アナログ -- 阻害剤ではないので除外
NATURAL_NUCLEOTIDES = frozenset({"ATP", "ADP", "AMP", "ANP", "ACP", "AGS", "GTP", "GDP", "GNP"})
MW_RANGE = (200.0, 700.0)

all_codes = sorted({code for codes in entry_nonpoly_codes.values() for code in codes})

_CHEM_COMP_QUERY = """
query($ids: [String!]!) {
    chem_comps(comp_ids: $ids) { chem_comp { id formula_weight } }
}
"""

formula_weight_by_code = {}
code_batches = [all_codes[i : i + _BATCH_SIZE] for i in range(0, len(all_codes), _BATCH_SIZE)]
for batch in tqdm(code_batches, desc="fetching chem_comp formula weights", unit="batch"):
    resp = requests.post(RCSB_GRAPHQL_API, json={"query": _CHEM_COMP_QUERY, "variables": {"ids": batch}}, timeout=60)
    resp.raise_for_status()
    for entry in resp.json()["data"]["chem_comps"]:
        cc = entry["chem_comp"]
        formula_weight_by_code[cc["id"]] = cc["formula_weight"]

print(f"formula weight resolved for {len(formula_weight_by_code)} of {len(all_codes)} distinct codes")

In [ ]:
def _is_inhibitor_code(code):
    if code in SOLVENT_AND_IONS or code in NATURAL_NUCLEOTIDES:
        return False
    mw = formula_weight_by_code.get(code)
    return mw is not None and MW_RANGE[0] <= mw <= MW_RANGE[1]


inhibitor_codes_by_entry = {
    entry_id: [c for c in codes if _is_inhibitor_code(c)] for entry_id, codes in entry_nonpoly_codes.items()
}
holo_entry_ids = [entry_id for entry_id, codes in inhibitor_codes_by_entry.items() if codes]
print(f"{len(holo_entry_ids)} of {len(all_entry_ids)} CDK2 entries have >=1 inhibitor-like ligand "
      f"(non-solvent/ion, non-nucleotide, {MW_RANGE[0]}-{MW_RANGE[1]} g/mol)")

### 阻害剤コードの出現頻度(全522構造ベース、ダウンロード不要)

構造ファイルを1件もダウンロードせずに、どのケミカルコードが繰り返し使われているかを集計できる
(前段のGraphQLレスポンスだけで完結する)。

In [ ]:
from collections import Counter

import pandas as pd

code_counts = Counter(code for codes in inhibitor_codes_by_entry.values() for code in codes)
top_codes_df = pd.DataFrame(code_counts.most_common(20), columns=["code", "n_structures"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_codes_df["code"][::-1], top_codes_df["n_structures"][::-1], color="teal")
ax.set_xlabel("n structures containing this inhibitor-like ligand code")
ax.set_title(f"CDK2: most frequently co-crystallized inhibitor codes (of {len(code_counts)} distinct)")
plt.tight_layout()
plt.show()
top_codes_df

### 阻害剤結合構造を、報告論文の被引用数でランク付けする

「過去の研究でよく使われた構造」の代理指標として、その構造を報告した論文([PubMed](
https://pubmed.ncbi.nlm.nih.gov/)ID経由)の被引用数を[NCBI iCite](https://icite.od.nih.gov/)
APIで取得する。同じ論文が複数のCDK2構造を報告していることも多いので、PubMed IDを重複排除して
から問い合わせる(iCiteは無料・無登録で使え、PMID一覧を渡すと被引用数を含むメタデータを返す)。

In [ ]:
ICITE_API = "https://icite.od.nih.gov/api/pubs"

holo_pubmed_ids = sorted(
    {
        entry_citation[entry_id].get("pdbx_database_id_PubMed")
        for entry_id in holo_entry_ids
        if entry_citation[entry_id].get("pdbx_database_id_PubMed")
    }
)

citation_count_by_pubmed = {}
pubmed_batches = [holo_pubmed_ids[i : i + _BATCH_SIZE] for i in range(0, len(holo_pubmed_ids), _BATCH_SIZE)]
for batch in tqdm(pubmed_batches, desc="fetching iCite citation counts", unit="batch"):
    resp = requests.get(ICITE_API, params={"pmids": ",".join(str(p) for p in batch)}, timeout=60)
    resp.raise_for_status()
    for rec in resp.json()["data"]:
        citation_count_by_pubmed[int(rec["_id"])] = rec.get("citation_count")

print(f"citation counts resolved for {len(citation_count_by_pubmed)} of {len(holo_pubmed_ids)} distinct PubMed IDs "
      f"cited by {len(holo_entry_ids)} inhibitor-bound entries")

In [ ]:
landscape_rows = []
for entry_id in holo_entry_ids:
    citation = entry_citation[entry_id]
    pubmed_id = citation.get("pdbx_database_id_PubMed")
    landscape_rows.append(
        {
            "entry_id": entry_id,
            "resolution": all_resolutions.get(entry_id),
            "inhibitor_codes": inhibitor_codes_by_entry[entry_id],
            "pubmed_id": pubmed_id,
            "citation_count": citation_count_by_pubmed.get(pubmed_id) if pubmed_id else None,
            "year": citation.get("year"),
            "title": citation.get("title"),
        }
    )

landscape_df = pd.DataFrame(landscape_rows)
landscape_df["citation_count_sort_key"] = landscape_df["citation_count"].fillna(-1)
landscape_df = landscape_df.sort_values(
    ["citation_count_sort_key", "resolution"], ascending=[False, True]
).drop(columns="citation_count_sort_key").reset_index(drop=True)

landscape_df.head(20)

### 被引用数上位の構造をダウンロードする

上位`TOP_N`件(被引用数優先、同点は高解像度優先)を実際にダウンロードする。`fpocket`は legacy
PDB形式を要求するので`filetype="pdb"`を指定する。`chem.rcsb.download_structures`はPDBエント
リIDのリストを渡すと、ターゲット解決や解像度フィルタを一切介さずその構造だけを直接取得する。
RCSBは一部の(主に大きい/新しい)エントリでlegacy PDB形式を提供しなくなっており、その場合は
警告付きで黙ってスキップされる(例外にはならない)ため、`TOP_N`より広めの候補プール
(`CANDIDATE_POOL`)をダウンロード対象にし、実際にPDBファイルが得られたものだけを順位順に
`TOP_N`件残す。

In [ ]:
import os

from chem import rcsb

TOP_N = 20
CANDIDATE_POOL = 40  # oversized -- some candidates may lack a legacy PDB file, see markdown above
candidate_entries = landscape_df["entry_id"].head(CANDIDATE_POOL).tolist()

rcsb.download_structures(candidate_entries, outdir="cdk2_data", filetype="pdb")
selected_entries = [
    e for e in candidate_entries if os.path.exists(os.path.join("cdk2_data", f"{e}.pdb"))
][:TOP_N]

print(f"{len(selected_entries)} of the top {CANDIDATE_POOL} citation-ranked candidates have a usable "
      f"legacy PDB file; using the top {TOP_N} of those")
landscape_df[landscape_df["entry_id"].isin(selected_entries)][
    ["entry_id", "resolution", "inhibitor_codes", "citation_count", "year", "title"]
]

## 2. ポケット

### 代表構造でのポケット検出

最多被引用の構造を1つ選び、`chem.protein.find_pocket`にかける。自動検出(構造中最大の非溶媒/
非イオンHETATM群を採用)に任せると、狙った阻害剤ではなく別のHETATM群(糖鎖修飾やたまたま大きい
結晶化添加剤)を拾ってしまう場合があるため、1で特定済みの阻害剤コードを`ligand`引数に明示的に
渡す。

In [ ]:
import os

from chem import protein

representative_entry = selected_entries[0]
representative_code = inhibitor_codes_by_entry[representative_entry][0]
representative_path = os.path.join("cdk2_data", f"{representative_entry}.pdb")

print(f"{representative_entry}: ligand={representative_code}, "
      f"citation_count={citation_count_by_pubmed.get(entry_citation[representative_entry].get('pdbx_database_id_PubMed'))}, "
      f"title={entry_citation[representative_entry].get('title')!r}")

pocket = protein.find_pocket(representative_path, ligand=representative_code)
{k: v for k, v in pocket.items() if k not in ("residues", "spheres", "info")}

### ポケットの可視化

`cdk20_pocket.ipynb`と同じ構成 -- 半透明のcartoon全体の上に、ポケット裏打ち残基をstickで、
fpocketのアルファ球(キャビティの形状/体積そのもの)を半透明球で重ねて表示する。

In [ ]:
import py3Dmol

with open(representative_path) as f:
    pdb_text = f.read()

top_resnums = sorted({r["resnum"] for r in pocket["residues"]})

view = py3Dmol.view(width=650, height=500)
view.addModel(pdb_text, "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})
view.addStyle({"hetflag": True}, {"stick": {"colorscheme": "yellowCarbon"}})
view.addStyle({"resi": top_resnums}, {"stick": {"colorscheme": "orangeCarbon"}})
for sphere in pocket["spheres"]:
    view.addSphere(
        {
            "center": {"x": sphere["x"], "y": sphere["y"], "z": sphere["z"]},
            "radius": sphere["radius"] * 0.4,
            "color": "cyan",
            "opacity": 0.35,
        }
    )
view.zoomTo({"resi": top_resnums})
view.show()

### 複数構造にわたるポケット裏打ち残基の保存性

被引用数上位5構造それぞれで、その構造自身の阻害剤コードを明示して`find_pocket`を実行し、
共結晶リガンドの化学構造が違っても裏打ち残基(`{chain, resnum, icode}`)がどれだけ共通してい
るかを確認する。いずれも同一のCDK2キナーゼドメインのATPポケットを狙った阻害剤結合構造である
はずなので、単純な解像度優先選定(参考: 別途試したところ裏打ち残基の共通部分が0件になった)
より高い保存性が期待できる。

In [ ]:
N_POCKET_ENTRIES = 5

pocket_residue_sets = {}
pocket_codes_used = {}
for entry_id in selected_entries[:N_POCKET_ENTRIES]:
    path = os.path.join("cdk2_data", f"{entry_id}.pdb")
    code_ = inhibitor_codes_by_entry[entry_id][0]
    p = protein.find_pocket(path, ligand=code_)
    pocket_residue_sets[entry_id] = {(r["chain"], r["resnum"], r["icode"]) for r in p["residues"]}
    pocket_codes_used[entry_id] = code_

common_residues = set.intersection(*pocket_residue_sets.values())
print(f"residues lining the pocket in all {N_POCKET_ENTRIES} structures: {len(common_residues)}")
for entry_id, residues in pocket_residue_sets.items():
    print(f"  {entry_id} (ligand: {pocket_codes_used[entry_id]}): {len(residues)} lining residues")
sorted(common_residues, key=lambda r: r[1])

## 3. ChEMBL活性データとの突合

### CDK2の高活性化合物を取得する

`example_chembl.ipynb`/`cdk_paralogs_active_compounds.ipynb`と同じパターン
(`normalize_smiles=True`でChEMBL Structure Pipelineによる標準化・脱塩+重複化合物の集約、
`mw=[250, 650]`でドラッグライクな分子量範囲に絞る)をCDK2単独に適用する。

In [ ]:
from chem import chembl

chembl.download_activities(
    "CDK2_HUMAN",
    mw=[250, 650],
    normalize_smiles=True,
    output="cdk2_chembl_activities.tsv",
)
activities_df = pd.read_csv("cdk2_chembl_activities.tsv", sep="\t")
len(activities_df)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(activities_df["pchembl_mean"], bins=40, color="darkorange", edgecolor="white")
ax.set_xlabel("pchembl_mean")
ax.set_ylabel("n compounds")
ax.set_title(f"CDK2_HUMAN: potency distribution across {len(activities_df)} unique compounds")
plt.tight_layout()
plt.show()

### 最高活性の化合物

`pchembl_mean`降順トップ10を、`example_chembl.ipynb`と同じ`MolsToGridImage`で描画する。

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import MolsToGridImage

TOP_N_COMPOUNDS = 10
top_compounds_df = activities_df.sort_values("pchembl_mean", ascending=False).head(TOP_N_COMPOUNDS)
display(top_compounds_df[["parent_chembl_id", "pchembl_mean", "n", "smiles"]])

mols = [Chem.MolFromSmiles(smi) for smi in top_compounds_df["smiles"]]
legends = [
    f"{cid} pchembl_mean={v:.2f}"
    for cid, v in zip(top_compounds_df["parent_chembl_id"], top_compounds_df["pchembl_mean"])
]
MolsToGridImage(mols, legends=legends, molsPerRow=5, subImgSize=(220, 200))

### 共結晶阻害剤はChEMBLの高活性化合物と一致するか

構造ランドスケープで集計した頻出阻害剤コード(全522構造ベースの上位30件)を、それぞれ
`chem.ligand.load_ligand`でPDB Chemical Component Dictionaryのテンプレートを使った正準SMILES
に変換し(共有結合修飾残基など、テンプレートが解決できないコードは`ValueError`でスキップ)、
ChEMBLデータセットの正準SMILESと突き合わせる。両方ともRDKitの`Chem.CanonSmiles`で正規化して
から比較するので、記法の違い(原子順序など)による見落としを避けられる。上位20件に既にダウン
ロード済みでないコードは、突合のためだけにその1構造を追加でダウンロードする。

In [ ]:
from rdkit import RDLogger

from chem import ligand

RDLogger.DisableLog("rdApp.*")  # load_ligand's template search is chatty on misses

def _canon(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.CanonSmiles(smiles) if mol is not None else None

chembl_canon_to_row = {}
for _, row in activities_df.iterrows():
    c = _canon(row["smiles"])
    if c is not None:
        chembl_canon_to_row[c] = row

TOP_CODES_FOR_CROSSREF = 30
matches = []
for code_, n_structures in code_counts.most_common(TOP_CODES_FOR_CROSSREF):
    # Try every entry carrying this code until one actually yields a legacy PDB file --
    # RCSB no longer serves the legacy PDB format for some (typically large/recent) entries,
    # so the first candidate isn't guaranteed to be downloadable.
    example_entry_id = None
    example_path = None
    for candidate in (e for e, codes in inhibitor_codes_by_entry.items() if code_ in codes):
        path = os.path.join("cdk2_data", f"{candidate}.pdb")
        if not os.path.exists(path):
            rcsb.download_structures([candidate], outdir="cdk2_data", filetype="pdb")
        if os.path.exists(path):
            example_entry_id, example_path = candidate, path
            break
    if example_entry_id is None:
        continue  # no entry with this code has a legacy PDB file available

    try:
        mol = ligand.load_ligand(example_path, code_)
    except ValueError:
        continue
    canon = Chem.MolFromSmiles(Chem.MolToSmiles(mol))
    if canon is None:
        continue
    canon_smiles = Chem.MolToSmiles(canon)
    chembl_row = chembl_canon_to_row.get(canon_smiles)
    if chembl_row is not None:
        matches.append(
            {
                "ligand_code": code_,
                "n_structures": n_structures,
                "example_entry_id": example_entry_id,
                "parent_chembl_id": chembl_row["parent_chembl_id"],
                "pchembl_mean": chembl_row["pchembl_mean"],
            }
        )

matches_df = pd.DataFrame(matches).sort_values("pchembl_mean", ascending=False) if matches else pd.DataFrame(
    columns=["ligand_code", "n_structures", "example_entry_id", "parent_chembl_id", "pchembl_mean"]
)
print(f"{len(matches_df)} of the top {TOP_CODES_FOR_CROSSREF} co-crystallized inhibitor codes "
      f"also appear in the ChEMBL {len(activities_df)}-compound set")
matches_df

## まとめ

CDK2 (`CDK2_HUMAN`) の全522 PDB構造を対象に、単純な解像度ではなく「実際の阻害剤が結合してい
るか(分子量フィルタ+天然ヌクレオチド除外)」と「その構造を報告した論文がどれだけ引用されて
いるか(NCBI iCite)」の2軸で評価し、`landscape_df`として横断的に整理した:

- **構造ランドスケープ**: `holo_entry_ids`が阻害剤結合構造、`landscape_df`が被引用数順のラン
  キング。上位`TOP_N`件を`cdk2_data/`にダウンロード済み。
- **ポケット**: 被引用数上位5構造それぞれで、自動検出ではなく実際の阻害剤コードを明示して
  `chem.protein.find_pocket`を実行し、裏打ち残基の保存性(`common_residues`)を確認した --
  ドッキングのポケット定義に流用できる。
- **活性データ**: `chem.chembl.download_activities`で取得した高活性化合物のうち、実際に共結晶
  構造として存在するものを正準SMILES一致で特定した(`matches_df`) -- 構造ベース設計の起点として
  そのまま3D配座が使える化合物の候補になる。

次の一手としては、`matches_df`に挙がった化合物を基準構造として`chem.protein.align`で他の
holo構造を重ね合わせ、ポケット内での結合様式の違いをSAR解析に繋げる、あるいは共結晶構造が
無いトップ活性化合物を`chem.protein.compute_transform`/ドッキングで代表ポケットに配置する、
といった方向が考えられる。